In [ ]:
%%html
<style>
    h1 {color:darkblue}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>


# Outline
0. Part 0 — Intro/Setup
1. **Part 1 — OpenAI Python APIs**
   * 01-01. **Text Generation Via the Responses API**
   * 01-02. Speech Recognition, Speech Synthesis, and Closed Captions
   * 01-03. Images: Generation and Style Transfer
   * 01-04. Content Moderation
   * 01-05. Generating Code with a Codex Model and the Responses API
2. Part 2 — OpenAI Agents SDK
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Text Generation Via the Responses API
* Apps that interact with OpenAI's GPT models to generate text responses based on text prompts

## Examples/Techniques this notebook:
* Shared setup: imports, API key assumption, and one `OpenAI` client object
* Summarizing a transcript and extracting key points
    * Rendering Markdown responses and streaming generated text
* **\[Python Fundamentals\]** Sentiment analysis of a presentation transcript
* Vision: Accessible image descriptions from image inputs
* Language detection and translation
* Named entity recognition (NER) with structured outputs
    * Three structured-output approaches
        * Prompt-only JSON
        * Strict JSON Schema
        * Pydantic typed objects
---


## Nondeterministic Responses
* Responses returned by genAIs are nondeterministic
    * Typically change from request to request
    * Even if your prompts are identical
    * Even when submitted to the same genAI
* Could be problematic for some applications
---

# Creating the `OpenAI` Client Object

In [ ]:
from openai import OpenAI 

In [ ]:
client = OpenAI()

* Assumes API key is in `OPENAI_API_KEY` environment variable

* If you used a different environment variable name, replace with
>```python
> client = OpenAI(api_key=os.environ.get("CUSTOM_ENV_NAME"))
>```

* If you store your key elsewhere, such as in a file
>```python
> import keys
> client = OpenAI(api_key=keys.OPENAI_API_KEY)
>```
---

# Common Imports for This Notebook

In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown                              

--- 

# Example 1 — Text Summarization and Streaming Responses
* Analyze a body of text and produce a concise version capturing key ideas
    * Quickly summarize articles, academic papers, book chapters or sections, emails, reports, study materials, product reviews, …
* Summarize a transcript as an abstract paragraph.
* Extract key points.
* Render Markdown.
* Stream a combined response as it is generated.
* `transcript.txt` (located in the `resources` subfolder) 
    * Created from introduction to Lesson 1's Test-Drives from my **Python Fundamentals** video course

### Loading the Presentation Transcript


In [ ]:
transcript_path = Path('resources') / 'transcript.txt'

In [ ]:
transcript = transcript_path.read_text()

In [ ]:
print(transcript)

### Requesting a Response That Produces a Summary Abstract Paragraph


In [ ]:
model_instructions = """Given a Python technical presentation's
transcript, create a summary abstract paragraph. Use straightforward
sentences. Spell language features and function names correctly. 
Avoid abbreviations. Do not refer to the speaker."""

In [ ]:
response = client.responses.create(model='gpt-5.4-nano', 
    instructions=model_instructions, input=transcript)

### Displaying the Response


In [ ]:
print(response.output_text)

### Token Counting and Cost Awareness
* Cost is based on input tokens and output tokens
* `response.usage` contains this info
* [Model pricing details](https://openai.com/api/pricing/ )
* [Your Usage Dashboard @ https://platform.openai.com/usage](https://platform.openai.com/usage)

In [ ]:
print(f'{response.usage.input_tokens=}\n{response.usage.output_tokens=}')

### Other GPT Models
* `'gpt-5.5'`
  * Current most powerful "thinking" model
* `'gpt-5.4-mini'`
  * Faster, cheaper; some "thinking"/planning before responding
* All `gpt-5.x` models can generate text, images and code

### Extracting Key Points


In [ ]:
model_instructions = """Given a Python technical presentation's
transcript, present a numbered list of the top 5 key points. 
Use concise, direct sentences and avoid abbreviations."""

In [ ]:
response = client.responses.create(model='gpt-5.4-nano', 
    instructions=model_instructions, input=transcript)

In [ ]:
print(response.output_text)

### Markdown-Formatted Responses


In [ ]:
Markdown(response.output_text)

## Streaming a Response

* Preceding calls wait for **complete response**
* `client.responses.create` 
    * `stream=True` to return an **event stream** of incremental updates
    * Like chatbots streaming their responses 
* `ResponseTextDeltaEvent.delta` 
    * Small text chunk rather than full `.output_text`
* Accumulate deltas into a string 
* `IPython.display.display(..., display_id=True)`
    * Returns handle for **updating a single output cell** rather than printing many separate lines
* `output_handle.update(Markdown(...))`
    * Re-render same output area in place as each chunk arrives

### Streaming a Combined Summary and Key Points Response


In [ ]:
model_instructions = """Given a Python technical presentation's transcript:

1. Write a summary abstract paragraph using straightforward sentences.
   Spell language features and function names correctly. Avoid
   abbreviations. Do not refer to the speaker.

2. Then present a numbered list of the top 5 key points using concise,
   direct sentences. Avoid abbreviations.

Format your response using Markdown with clear section headings."""

In [ ]:
accumulated_text = ''
output_handle = display(Markdown(''), display_id=True)

with client.responses.create(model='gpt-5.4-nano',
        instructions=model_instructions, input=transcript,
        stream=True) as stream:
    for event in stream:
        if event.type == 'response.output_text.delta':
            accumulated_text += event.delta
            output_handle.update(Markdown(accumulated_text))

---

# Example 2 — Sentiment Analysis
* Reuse transcript loaded in the summarization example and ask the model to classify and explain the sentiment
    * Positive, neutral or negative

In [ ]:
model_instructions = """You are a sentiment-analysis expert. 
Determine the provided transcript's sentiment. Explain your analysis."""

In [ ]:
response = client.responses.create(model='gpt-5.4-mini', 
    instructions=model_instructions, input=transcript)

In [ ]:
Markdown(response.output_text)

---
# Example 3 — Accessible Image Descriptions
* Responses API and vision capabilities
* World Wide Web Consortium (W3C): 
    * Web should be accessible to everyone, regardless of ability
* Accessible image descriptions
    * For people who are blind or have low vision
* Multimodal application with image and text inputs
  * Uploads an image
  * Requests a detailed image description based on the World Wide Web Consortium (W3C) accessible-image guidelines

---

## Import Function `describe_image`
* [Open describe_image.py](./source/describe_image.py)
* [Open util.py](./source/util.py)

In [ ]:
from source.describe_image import describe_image

--- 

### Getting an Accessible Description of a Scenic Aruba Sunset Photo
<img src="./resources/sunset.jpg" width="75%"/>


In [ ]:
description = describe_image(client,
    'Provide an accessible description of this Aruba sunset.', 
    Path('resources') / 'sunset.jpg')

In [ ]:
Markdown(description)

---

### Getting an Accessible Description of a Word Cloud
<img src="./resources/RomeoAndJulietHeart.png" width="75%"/>


In [ ]:
description = describe_image(client,
    """Provide an accessible description of this 
    Romeo and Juliet word cloud.""", 
    Path('resources') / 'RomeoAndJulietHeart.png')

In [ ]:
Markdown(description)

## Creating the Local Image File's URL
* Images are stored as bytes
* When transmitted over the Internet between systems that handle the raw bytes differently, the bytes can get corrupted
* Those systems typically understand characters
* Raw bytes often encoded into character representations
* Base64 — named for the Base64 alphabet's 64 characters
  * 52 letters (`A`–`Z` and `a`–`z`)
  * 10 digits (`0`–`9`)
  * `+` and `/`
* All printable, easily understood by world's computer systems
---

## Data URL Format
* Local images passed to Responses API must be Base64-encoded and embedded in a URL with the format
  * `data:image/imageFormat;base64,base64EncodedImageString`
    * `imageFormat` in our example is `png` or `jpeg`
    * `image/imageFormat` is the MIME type of the image
* `create_data_url` utility function (in [util.py](./source/util.py))
  * Base64-encodes the image
  * Returns a properly formatted `data:` URL
---

## Passing the Image to the Responses API
* [Open describe_image.py](./source/describe_image.py)
* Responses API multimodal inputs use different input format
* API requires JSON array of JSON objects, each with an optional text prompt and the URL of an image to analyze
* In Python, Multimodal inputs require a list of dictionaries
  * Auto formatted by `responses.create` as JSON
  * We analyze only one image at a time, so the list in this example contains one dictionary

---

## Accessible Image Descriptions — Dictionary Format

* Each dictionary requires two key-value pairs:
  * `'role': 'user'` — input includes the user's data on which model operates
  * `'content'` key's value — list of data to provide to model as input
  * When uploading an image, the `'content'` key's value is a list of dictionaries
* Optional: Prompt dictionary with context or instructions
  * `'type': 'input_text'`
  * `'text'` key with the prompt string as its value
* Image dictionary
  * `'type': 'input_image'`
  * `'image_url'` — `data:` URL containing Base64-encoded image

---
# Example 4 — Language Detection and Translation
* genAIs can translate between languages
* Wide range of uses, including:
  * Cross-language communication: Facilitating conversations between people speaking different languages
  * Global business operations: Translating documents, websites and customer communications for international markets
  * Tourism: Helping travelers communicate in foreign countries
  * Education: Enabling students and researchers to access academic content in different languages
  * Healthcare: Assisting healthcare providers in understanding patients who speak different languages
  * Diplomatic relations: Supporting negotiations and communications between countries, such as at the United Nations
  * Entertainment: Translating subtitles, scripts and media for global audiences

## Translation Example
* Use Responses API to translate text among languages
* Autodetect the text's source language then translate the text to a specified target language
  * Translate English to Spanish and Japanese
  * Translate the Spanish and Japanese text back to English
* Well-defined task that does not require significant reasoning, so we’ll use gpt-5.4-mini

## Translate English to Spanish and Japanese
* Automatically figures out source language
* [Open translate.py](./source/translate.py)

In [ ]:
from source.translate import translate

In [ ]:
english_text = (
    'Today was a beautiful day. Tomorrow looks like bad weather.')

In [ ]:
spanish_text = translate(client, english_text, 'Spanish')
spanish_text

In [ ]:
japanese_text = translate(client, english_text, 'Japanese')
japanese_text

### Saving Translated Text for the Speech Notebook
* Save in `resources/outputs` so `01-02-speech.ipynb` can reuse them later

In [ ]:
translated_text_dir = Path('resources') / 'outputs'
translated_text_dir.mkdir(parents=True, exist_ok=True)

spanish_text_path = translated_text_dir / 'spanish_text.txt'
japanese_text_path = translated_text_dir / 'japanese_text.txt'

spanish_text_path.write_text(spanish_text, encoding='utf-8')
japanese_text_path.write_text(japanese_text, encoding='utf-8')

spanish_text_path, japanese_text_path

### Translating Spanish and Japanese Back to English


In [ ]:
translate(client, spanish_text, 'English')

In [ ]:
translate(client, japanese_text, 'English')

In [ ]:
english_text

---
# Example 5 — Named Entity Recognition (NER) and Structured Outputs
* Locate and categorize entities such as dates, times, quantities, places, people, things, organizations and more
* Structured JSON output
    * Make results easier to process programmatically
    * We'll provide the exact JSON response format we want
    * Use the json module to process and display the results

---

### NER Tag Set
* Initial tests gave different results for each run
  * Sometimes using tags
  * Sometimes using words
* For more deterministic results, told model to use OntoNotes Named Entity Tag Set
  * spaCy uses this in its NER results

---

### JSON Output for Named Entities
* Want model to return response as a JSON object with an `"entities"` key and an array of JSON objects as its value
* Each object in the array contains:
  * `"text"` attribute for a named entity
  * `"tag"` attribute indicating that named entity's type
* Include this as part of request

```json
{
   "entities": [
      {
         "text": "Entity name",
         "tag": "Entity type"
      }
   ]
}
```

---


### Loading the Text
* We'll perform NER on the text in the file web.txt, which contains a paragraph about the World Wide Web from Chapter 1
* This file is located in the resources subfolder with this chapter's examples


In [ ]:
path = Path('resources') / 'web.txt'

In [ ]:
text = path.read_text()

In [ ]:
text

---

### Performing NER on the Loaded Text
* Submit a request to the Responses API containing:
  * Model to use (`'gpt-5.4-mini'`)
  * Instructions telling the model it's an expert in named entity recognition using the OntoNotes Named Entity Tag Set
  * Input prompt that asks the model to analyze the text's named entities and return them in JSON format shown in the prompt


In [ ]:
response = client.responses.create(
    model='gpt-5.4-mini',
    instructions="""You are an expert in named entity 
        recognition with the OntoNotes Named Entity Tag Set.""",
    input="""Analyze the supplied text and extract its named
        entities. Return only JSON in the following format:
            {
              "entities": [
                {
                  "text": "Entity name",
                  "tag": "Entity type"
                }
              ]
            }   
        Text to analyze:
        """ + text
)

### Deserializing the response
* Use the `json` module's `loads` ("load string") function to deserialize the JSON in its string argument, creating the dictionary json_response
* For readability, use the `json` module's `dumps` ("dump string") function to create a structured string representation of the named entities
* The output shows the named entities categorized by type 

In [ ]:
json_response = json.loads(response.output_text)

In [ ]:
print(json.dumps(json_response, indent=2))

## Three Approaches to Structured Responses
* Approach 1 — Prompt-Based (No Schema Enforcement) 
* Approach 2 — Raw JSON Schema (API-Enforced)
* Approach 3 — Pydantic (Typed Python Objects)
---


### Approach 1 — Prompt-Based (No Schema Enforcement)
This is what we just did.

**Pros**
* Zero setup — describe the format in `instructions` or `input` string
* Works with any model; no extra SDK parameters needed
* Easy to adjust the schema by editing a string

**Cons**
* No enforcement — the model may deviate from the requested shape or return malformed JSON
* `json.loads()` can raise `JSONDecodeError` at runtime if the model misbehaves
* Extra or missing fields cause bugs 
* All validation is your responsibility

---


### Approach 2 — Raw JSON Schema (API-Enforced)

**Pros**
* API guarantees response matches schema — `json.loads()` always succeeds
* Schema is explicit and reusable
* Ideal when schema is constructed dynamically at runtime or read from an external source

**Cons**
* Writing JSON schema by hand is error-prone
* Result is still a plain `dict` and `list` objects 
    * no Python type checking or IDE autocomplete
* `additionalProperties: false` and `required` arrays must be set on every object
    * Otherwise, OpenAI API rejects the schema
---

#### JSON Schema


In [ ]:
ner_schema = {
    'type': 'object',
    'properties': {
        'entities': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'text': {'type': 'string'},
                    'tag':  {'type': 'string'}
                },
                'required': ['text', 'tag'],
                'additionalProperties': False
            }
        }
    },
    'required': ['entities'],
    'additionalProperties': False
}

```python
ner_schema = {
    # top-level output must be a JSON object: { ... }
    'type': 'object',

    # object may contain these named properties
    'properties': {
        # nne property named "entities"
        'entities': {
            # "entities" must be an array/list
            'type': 'array',

            # each array item must follow this schema
            'items': {
                # each array item must be an object
                'type': 'object',

                # each object may contain these properties
                'properties': {
                    # actual text found in the input
                    'text': {'type': 'string'},

                    # entity label/category
                    'tag':  {'type': 'string'}
                },

                # every object must include both fields
                'required': ['text', 'tag'],

                # no extra fields allowed inside each entity object
                'additionalProperties': False
            }
        }
    },

    # top-level object must include "entities"
    'required': ['entities'],

    # no extra top-level fields allowed
    'additionalProperties': False
}
```

#### Calling `responses.create` with `text={'format': ...}`


In [ ]:
response = client.responses.create(
    model='gpt-5.4-mini',
    instructions="""You are an expert in named entity
        recognition with the OntoNotes Named Entity Tag Set.""",
    input='Analyze the supplied text and extract its named entities.\n\n' + text,
    text={
        'format': {
            'type': 'json_schema',
            'name': 'NERResult',
            'schema': ner_schema,
            'strict': True # output must confirm to this schema
        }
    }
)

#### Deserializing — Always Succeeds Because the API/Model Enforce the Schema

In [ ]:
json_response = json.loads(response.output_text)
print(json.dumps(json_response, indent=2))

---

### Approach 3 — Pydantic (Typed Python Objects)
* https://docs.pydantic.dev
* Structured data validation 
* Pydantic enforces types and validates data at runtime using standard Python type hints

#### Pros for Structured Outputs
* Define the schema once as Pydantic `BaseModel` 
* SDK generates JSON Schema automatically from Pydantic model
* `response.output_parsed` is a **typed Python object** — no `json.loads()` needed
* IDE autocomplete and static type checking support
* **Best for production code**

#### **Cons** for Structured Outputs
* Requires Pydantic (but this is already an OpenAI SDK dependency)
* Model must be defined ahead of time — less flexible for truly dynamic schemas
* More initial setup

### Defining a Pydantic Model
* `BaseModel`
    * Superclass for defining a schema
    * Fields declared as annotated class attributes

In [ ]:
from pydantic import BaseModel
from typing import Optional

class Person(BaseModel):
    name: str
    age: int
    email: Optional[str] = None

* `pydantic.Field(...)` can be used to add descriptive metadata to fields
    * `description=`, `default=`, `gt=`, `lt=`, ...
    * `description` becomes part of JSON schema model uses when generating structured output

##### Pydantic validates and coerces types on construction

In [ ]:
p1 = Person(name="Alice", age="30")   # age coerced from str to int 

In [ ]:
p2 = Person(name="Alice", age="abc")  # raises ValidationError as "abc" is not convertable to int

#### JSON for a Pydantic Object and Schema
* `model_object.model_dump()` 
    * Returns a `dict` representation of the schema
* `model_object.model_json_schema()` 
    * Returns the full JSON schema

In [ ]:
p1.model_dump()

In [ ]:
print(json.dumps(p1.model_json_schema(), indent=2))

#### Pydantic and the Responses API
* Pass a Pydantic model as `text_format=` to `client.responses.parse()`
    * SDK serializes it to JSON schema
    * SDK sends schema to model as a constraint
    * SDK deserializes model's JSON output into a validated Python object
        * `response.output_parsed`

#### Defining the Pydantic Models


In [ ]:
from pydantic import BaseModel 

class Entity(BaseModel):
    text: str
    tag: str

class NERResult(BaseModel):
    entities: list[Entity]

#### Calling `responses.parse` with `text_format=NERResult`

* Use `client.responses.parse` instead of `client.responses.create`
* Pass the Pydantic class as `text_format` — the SDK converts it to JSON Schema for you
* `response.output_parsed` is an `NERResult` instance — no `json.loads()` required


In [ ]:
response = client.responses.parse(
    model='gpt-5.4-mini',
    instructions="""You are an expert in named entity
        recognition with the OntoNotes Named Entity Tag Set.""",
    input='Analyze the supplied text and extract its named entities.\n\n' + text,
    text_format=NERResult 
)

#### Accessing the Typed Result


In [ ]:
result: NERResult = response.output_parsed

for entity in result.entities:
    print(f'{entity.tag:20} {entity.text}')

---

### Note: `output_parsed` is typed `Optional[ContentType]`
* Can return `None` if response contains no parsed output message
* Typically, not `None` when `text_format` is passed
* Defensive programming: Check that `result` is not `None`

---

### Summary of Pros/Cons 

| | Prompt-Based | Raw JSON Schema | Pydantic |
|---|---|---|---|
| **Schema enforced by API** | No | Yes | Yes |
| **Python type safety** | No | No | Yes |
| **`json.loads()` required** | Yes — can raise `JSONDecodeError` | Yes — always succeeds | No |
| **IDE autocomplete on result** | No | No | Yes |
| **Dynamic / runtime schemas** | Yes | Yes | Harder |
| **Best for** | Quick prototypes | External/dynamic schemas | Production code |

---

# Additional Info for Your Reference
## Error Handling — `RateLimitError`, `APIStatusError`, and Retries

* OpenAI Python SDK exceptions inherit from `openai.APIError`
* **`RateLimitError`** maps to HTTP `429`
    * Often means too many requests or tokens in a rate-limit interval
    * Back off and retry; avoid retrying indefinitely
* **`APIStatusError`**
    * Raised for non-success HTTP responses (`4xx` or `5xx`)
    * Exposes `.status_code`, `.response`, and `.request_id`
    * Use `str(e)` for the error message
* **`max_retries`**
    * Configures automatic short exponential-backoff retries
    * Default is `2`
    * Retries include connection errors, `408`, `409`, `429`, and `>=500`
    >```python
    ># configure automatic retry count 
    >client = openai.OpenAI(max_retries=5)
    >```

### Reference Links
* **Error codes guide:** https://developers.openai.com/api/docs/guides/error-codes
* **Python library — Handling errors:** https://github.com/openai/openai-python?tab=readme-ov-file#handling-errors
* **Python library — Retries:** https://github.com/openai/openai-python?tab=readme-ov-file#retries
* **Rate limits guide:** https://platform.openai.com/docs/guides/rate-limits

---

## `client.responses.create` — Common Parameters
| Parameter | Purpose |
|---|---|
| `model` | Selects the model to use |
| `instructions` | Sets system/developer-level behavior |
| `input` | Supplies the user prompt or input items |
| `max_output_tokens` | Caps generated output length and helps control cost |
| `temperature` | Adjusts output randomness/variety |
| `text` | Controls text output format, including structured JSON |

### Reference Link
* [Responses API — Create a model response](https://developers.openai.com/api/reference/resources/responses/methods/create)  


<hr/>
© 2026 by Deitel & Associates, Inc. All Rights Reserved.